# 3 · A physics agent: the nonlinear pendulum

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MonicaCoraggioso/my-first-agents/blob/main/3-physics_agent_pendulum.ipynb)

**Goal:** apply the agent loop from notebook 2 to a physics problem. The agent must answer questions like:

> *"How does the period of the pendulum change from 10° to 80°? Compare it with the small-angle approximation."*

To do so, it **decides by itself** which simulations to run, measures the periods, compares them with the theoretical formula and explains the result.

**Same structure as notebook 2**

```
┌─► 1. Construct prompt   (agent rules + memory)
│   2. Generate response  (LLM picks the next action)
│   3. Parse response     (extract tool_name + args)
│   4. Execute action     (run the physics Python function)
│   5. Update memory      (store action + result)
└── 6. Continue?          (stop on terminate() or max iterations)
```

**What changes compared with notebook 2:** only the **tools**.

| Notebook 2 (files) | Notebook 3 (physics) |
|---|---|
| `list_files()` | `simulate_pendulum(theta0_deg, length_m, t_max)` |
| `read_file(file_name)` | `measure_period(sim_id)` |
| | `small_angle_period(length_m)` |
| `terminate(message)` | `terminate(message)` |

**Key idea: reasoning vs computing**
- A small LLM like MiniCPM (2 billion parameters) is **unreliable at arithmetic** → we don't ask it to compute.
- The LLM **reasons and decides** (which simulation, with which parameters, how to interpret the numbers).
- The Python code (NumPy, SciPy) **computes** accurately.

**How this notebook is organised**
- Sections **Step 1 → Step 6** explain each part of the loop and define the code it needs (rules, parser, tools).
- The tools are **tested on their own** against the exact solution (with output).
- The **agent loop** at the end runs all six steps together; its output is a real run of MiniCPM, analysed in the last section.

## A bit of physics

Equation of motion of the simple pendulum (no friction), with $\theta$ the angle from the vertical, $L$ the length, $g = 9.81\ \mathrm{m/s^2}$:

$$\ddot\theta = -\frac{g}{L}\,\sin\theta$$

| | Small oscillations ($\theta \ll 1$) | General case |
|---|---|---|
| Approximation | $\sin\theta \approx \theta$ → **harmonic** oscillator | **nonlinear** equation |
| Period | $T_0 = 2\pi\sqrt{L/g}$, **independent of the amplitude** (isochronism) | $T(\theta_0) > T_0$, **grows with the amplitude** |
| How to compute it | formula | numerical integration (or elliptic integral) |

→ Using the tools, the agent should find that at 10° the period is almost equal to $T_0$, while at 80° it is about 14% longer.

## Glossary

| Term | Meaning in this notebook |
|---|---|
| **Agent** | LLM + loop that, iteration after iteration, chooses and executes actions until the task is done |
| **Tool** | Python function the agent can ask to run (e.g. `simulate_pendulum`) |
| **Action** | the LLM's request to use a tool: `{"tool_name": ..., "args": ...}` |
| **Environment** | what the tools act on: here, numerical simulations |
| **Memory** | the list of messages: task, actions and results |

**Prerequisites:** notebook 2 (agent loop). **Requirements:** `torch`, `transformers`, `accelerate`, `numpy`, `scipy` (in Colab: `!pip install -q transformers accelerate`, the rest is pre-installed); GPU recommended (*Runtime → Change runtime type → T4 GPU*); ~5 GB for the model.

## Setup

In [1]:
import json                       # read/write the agent's JSON actions
from typing import Dict, List     # type hints

import numpy as np                # numerical computing
from scipy.integrate import solve_ivp   # ODE integrator

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
# Load MiniCPM (~5 GB, downloaded on first run, then cached)
model_id = "openbmb/MiniCPM5-2B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",   # keep the precision the weights were saved in
    device_map="auto",    # use the GPU automatically when available
)

print("Model loaded successfully!")

Model loaded successfully!


In [3]:
def generate_response(messages: List[Dict]) -> str:
    """Send a list of chat messages to MiniCPM and return its reply as text."""
    # Format the messages with the model's chat template -> token IDs
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,   # add the "assistant's turn" marker
        enable_thinking=False,        # skip MiniCPM's hidden reasoning step
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    # Generate (greedy = deterministic); no gradients, we are not training
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
        )

    # Decode only the new tokens (drop the prompt)
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    return response

## Step 1 · Construct the prompt

```python
prompt = agent_rules + memory
```

| Part | What it contains | Why |
|---|---|---|
| `agent_rules` | **system** message: role, physics tools, constraints, response format | the agent always knows *what it can do* and *how to answer* |
| `memory` | user's task + actions already taken + results | the agent knows *what has already been computed* → decides the next step |

### Agent rules

We directly use the **improved** version from notebook 2:

- **Role** → *physics assistant that uses numerical tools*.
- **Tools** described as a **JSON schema** (description + typed parameters):
  - `simulate_pendulum(theta0_deg, length_m, t_max)` → integrates the nonlinear equation, returns an id `sim_id`
  - `measure_period(sim_id)` → period measured from that simulation
  - `small_angle_period(length_m)` → $T_0 = 2\pi\sqrt{L/g}$
  - `terminate(message)` → final answer to the user
- **Physics-specific constraints:**
  - *never do calculations in your head, use the tools* → avoids made-up numbers;
  - *one simulation per amplitude, then measure the period* → the right order of steps;
  - *in the final message, report the numbers and explain the physics*.
- **Format** → first think step by step, then the ```` ```action {...} ``` ```` block.

Prompt structure (**GAIL** framework):

| Letter | Part of the prompt | Here |
|---|---|---|
| **G**oals | goal / instructions | "You are a physics assistant…", constraints |
| **A**ctions | available actions | the JSON schema of the tools |
| **I**nformation | context information | the memory (task + results) |
| **L**anguage | response format | thoughts + ```` ```action ```` block |

In [4]:
# Agent rules: system message sent at the start of EVERY prompt
agent_rules = [{
    "role": "system",
    "content": """
You are a physics assistant that answers questions about the simple pendulum
by running numerical tools. You never do calculations in your head: you always use the tools.

Available tools:

```json
{
    "simulate_pendulum": {
        "description": "Numerically integrates the NONLINEAR pendulum equation theta'' = -(g/L) sin(theta), starting at rest from angle theta0. Returns a simulation id.",
        "parameters": {
            "theta0_deg": {"type": "number", "description": "Initial angle in degrees (between 0 and 180, excluded)."},
            "length_m": {"type": "number", "description": "Pendulum length in meters. Default 1.0."},
            "t_max": {"type": "number", "description": "Simulated time in seconds. Default 20.0."}
        }
    },
    "measure_period": {
        "description": "Measures the oscillation period of a previous simulation.",
        "parameters": {
            "sim_id": {"type": "string", "description": "Simulation id returned by simulate_pendulum, e.g. 'sim_1'."}
        }
    },
    "small_angle_period": {
        "description": "Period predicted by the small-angle (harmonic) approximation T0 = 2*pi*sqrt(L/g).",
        "parameters": {
            "length_m": {"type": "number", "description": "Pendulum length in meters. Default 1.0."}
        }
    },
    "terminate": {
        "description": "Ends the task and gives the final answer to the user.",
        "parameters": {
            "message": {"type": "string", "description": "Final answer: report the numbers and explain the physics."}
        }
    }
}
```

Rules:
- For each amplitude: first call simulate_pendulum, then measure_period with the returned sim_id.
- Use small_angle_period when the user asks for a comparison with the small-angle approximation.
- Use only one tool per response.
- When you have all the numbers, use "terminate" with a clear answer, in the same language as the user.

Important!!! Every response MUST have an action.
You must ALWAYS respond in this format:

<Stop and think step by step. Parameters map to args. Insert a rich description of your step by step thoughts here.>

```action
{
    "tool_name": "insert tool_name",
    "args": {...fill in any required arguments here...}
}
```"""
}]

### Memory

The memory is a list of messages. In the loop it **starts with only the user's task** and grows by two messages per iteration (see Step 5). Example after one iteration:

| # | Role | Content | Meaning |
|---|---|---|---|
| 1 | `user` | *How does the period change from 10° to 80°?…* | the task |
| 2 | `assistant` | thoughts + ```` ```action {"tool_name": "simulate_pendulum", "args": {"theta0_deg": 10}} ``` ```` | the chosen action |
| 3 | `user` | `{"result": "sim_1: theta0 = 10°, …"}` | the tool's **result** |

→ The logical next step for the agent would be `measure_period("sim_1")`.

## Step 2 · Generate the response

```python
response = generate_response(prompt)   # defined in Setup, called inside the loop
```

- Sends the prompt (rules + memory) to the LLM.
- The LLM **decides the next action** based on the context.
- Expected answer: step-by-step thoughts, then a ```` ```action ```` block.

## Interfacing the agent with the environment

```
LLM response ──► 3. parse ──► action {tool_name, args} ──► 4. execute ──► result ──► 5. memory
```

- The LLM only produces **text** → the interface turns it into a real **action** (here: a numerical simulation).

## Step 3 · Parse the response

**Same as notebook 2.** Goal: extract `tool_name` and `args` from the response.

- Find the ```` ```action ```` block (even if the thoughts come first) → `json.loads` → check that `tool_name` and `args` are there.
- Invalid response → `error` action with a message telling the LLM how to fix it.

| Problem | Returned action |
|---|---|
| valid JSON, but no `tool_name` / `args` | `error`: *"You must respond with a JSON tool invocation."* |
| invalid JSON | `error`: *"Invalid JSON response. …"* |

In [5]:
def extract_markdown_block(response: str, block_type: str = "json") -> str:
    """Return the content of the Markdown code block of type `block_type` (e.g. ```action).

    1. If there is a ```<block_type> block, return the text up to the next ```
       (works even if thoughts or other code blocks come first).
    2. Otherwise use the first code block of any type.
    3. If there is no code block, return the response unchanged.
    """
    marker = f"```{block_type}"

    # 1. Block with the right tag, e.g. "```action"
    if marker in response:
        after_marker = response.split(marker, 1)[1]
        return after_marker.split("```", 1)[0].strip()

    # 3. No code block at all
    if '```' not in response:
        return response

    # 2. First code block, without its language tag
    code_block = response.split('```')[1].strip()
    if code_block.startswith(block_type):
        code_block = code_block[len(block_type):].strip()
    return code_block


def parse_action(response: str) -> Dict:
    """Parse the LLM response into a structured action dictionary.

    Returns:
        {"tool_name": ..., "args": {...}} if the response is a valid tool call,
        otherwise an "error" action whose message tells the LLM how to fix it.
    """
    try:
        # 1. Keep only the content of the ```action block
        response = extract_markdown_block(response, "action")
        # 2. Text -> Python dict
        response_json = json.loads(response)
        # 3. Check that it is a proper tool invocation
        if isinstance(response_json, dict) and "tool_name" in response_json and "args" in response_json:
            return response_json
        else:
            return {"tool_name": "error", "args": {"message": "You must respond with a JSON tool invocation."}}
    except json.JSONDecodeError:
        # Not valid JSON at all
        return {"tool_name": "error", "args": {"message": "Invalid JSON response. You must respond with a JSON tool invocation."}}

## Step 4 · Execute the action

**Goal:** run the Python function that matches `tool_name`, with the parsed `args`.

First, the **physics tools**. Three design rules for a small LLM:

| Rule | Why | How |
|---|---|---|
| **Short text results** | every result goes into the memory, which is limited | `"sim_1: period = 2.0099 s"`, not 2000 points |
| **Heavy data stays in the code** | the LLM must not "carry" trajectories around | `simulations` dict; the LLM only sees `sim_id` |
| **Validated inputs** | the LLM may pass absurd values or strings | `float(...)`, range checks, clear error messages |

**How we measure the period:** `solve_ivp` with an **event** that records the exact times when $\theta = 0$ (crossing the vertical). Two consecutive crossings are half a period apart → $T = 2 \times$ (mean time between crossings).

In [6]:
G = 9.81            # gravitational acceleration [m/s^2]
simulations = {}    # simulation results live here (NOT in the LLM's memory)


def simulate_pendulum(theta0_deg: float, length_m: float = 1.0, t_max: float = 20.0) -> str:
    """Integrate the NONLINEAR pendulum equation  θ'' = -(g/L) sin θ.

    The pendulum starts at rest from angle theta0_deg. The trajectory is stored
    in `simulations` and only an id (e.g. "sim_1") is returned to the LLM.
    """
    # Arguments come from the LLM: convert and check them
    theta0_deg, length_m, t_max = float(theta0_deg), float(length_m), float(t_max)
    if not 0 < theta0_deg < 180:
        return "Error: theta0_deg must be between 0 and 180 degrees (excluded)."
    if length_m <= 0 or t_max <= 0:
        return "Error: length_m and t_max must be positive."

    # First-order system: y = [θ, ω]  ->  y' = [ω, -(g/L) sin θ]
    def rhs(t, y):
        return [y[1], -(G / length_m) * np.sin(y[0])]

    # Event: θ = 0 (crossing the vertical), used to measure the period
    def crosses_vertical(t, y):
        return y[0]

    sol = solve_ivp(
        rhs, [0, t_max], [np.radians(theta0_deg), 0.0],   # initial conditions: θ0, ω = 0
        events=crosses_vertical,
        rtol=1e-9, atol=1e-9,                             # high accuracy
    )

    sim_id = f"sim_{len(simulations) + 1}"
    simulations[sim_id] = {"theta0_deg": theta0_deg, "length_m": length_m, "solution": sol}
    return f"{sim_id}: theta0 = {theta0_deg:g}°, L = {length_m:g} m, simulated for {t_max:g} s"


def measure_period(sim_id: str) -> str:
    """Measure the period from the times the pendulum crosses the vertical."""
    if sim_id not in simulations:
        return f"Error: unknown sim_id '{sim_id}'. Available: {list(simulations)}"

    sim = simulations[sim_id]
    crossings = sim["solution"].t_events[0]          # times when θ = 0
    if len(crossings) < 3:
        return f"Error: {sim_id} is too short to measure the period. Simulate a longer t_max."

    period = 2 * np.mean(np.diff(crossings))         # 2 consecutive crossings = half a period
    return f"{sim_id}: theta0 = {sim['theta0_deg']:g}°, measured period = {period:.4f} s"


def small_angle_period(length_m: float = 1.0) -> str:
    """Period in the harmonic (small-angle) approximation: T0 = 2π √(L/g)."""
    length_m = float(length_m)
    if length_m <= 0:
        return "Error: length_m must be positive."
    return f"small-angle period T0 = {2 * np.pi * np.sqrt(length_m / G):.4f} s (L = {length_m:g} m)"

### Testing the tools (before giving them to the agent!)

An agent is only as reliable as its tools → we test them on their own, against the **exact solution** of the pendulum:

$$T(\theta_0) = \frac{4}{\omega_0}\,K\!\left(\sin^2\tfrac{\theta_0}{2}\right), \qquad \omega_0 = \sqrt{g/L}$$

where $K$ is the complete elliptic integral of the first kind (`scipy.special.ellipk`).

In [7]:
from scipy.special import ellipk

def exact_period(theta0_deg, length_m=1.0):
    """Exact period of the nonlinear pendulum (elliptic integral) - only for testing."""
    m = np.sin(np.radians(theta0_deg) / 2) ** 2
    return 4 / np.sqrt(G / length_m) * ellipk(m)

print(small_angle_period())
for angle in [10, 45, 80, 120]:
    print(simulate_pendulum(angle))
    sim_id = f"sim_{len(simulations)}"
    print("  ", measure_period(sim_id), f"| exact = {exact_period(angle):.4f} s")

simulations.clear()   # remove the test simulations: the agent starts from sim_1

small-angle period T0 = 2.0061 s (L = 1 m)
sim_1: theta0 = 10°, L = 1 m, simulated for 20 s
   sim_1: theta0 = 10°, measured period = 2.0099 s | exact = 2.0099 s
sim_2: theta0 = 45°, L = 1 m, simulated for 20 s
   sim_2: theta0 = 45°, measured period = 2.0863 s | exact = 2.0863 s
sim_3: theta0 = 80°, L = 1 m, simulated for 20 s
   sim_3: theta0 = 80°, measured period = 2.2819 s | exact = 2.2819 s
sim_4: theta0 = 120°, L = 1 m, simulated for 20 s
   sim_4: theta0 = 120°, measured period = 2.7541 s | exact = 2.7541 s


### Execution logic

As in notebook 2: `tool_name` → Python function.

| `tool_name` | What happens | Result |
|---|---|---|
| `simulate_pendulum` | `simulate_pendulum(**args)` | `{"result": "sim_1: ..."}` |
| `measure_period` | `measure_period(args["sim_id"])` | `{"result": "... period = ... s"}` |
| `small_angle_period` | `small_angle_period(**args)` | `{"result": "T0 = ... s"}` |
| `error` | (from the parser) | `{"error": message}` |
| `terminate` | print the final answer | stop (`break` in the loop) |
| anything else | unknown tool | `{"error": "Unknown action: ..."}` |

**New compared with notebook 2:**
- `**args` passes all the arguments at once: `{"theta0_deg": 80}` → `simulate_pendulum(theta0_deg=80)`; missing ones take their default value.
- `try/except`: if the LLM gets the arguments wrong (e.g. forgets one or invents one), instead of crashing, the error goes back to the LLM as the result → **the agent can correct itself**.

This logic runs **inside the agent loop** at the end of the notebook.

## Step 5 · Update the memory

**Same as notebook 2.** After each action, **two messages** are appended:

| Role | Content | Meaning |
|---|---|---|
| `assistant` | the LLM's `response` | the **intention** (chosen action) |
| `user` | `json.dumps(result)` | the **feedback**: what the tool produced |

- Results **always** come back with the `user` role.
- This way the agent "remembers" which simulations it ran and which periods it already measured.

```python
memory.extend([
    {"role": "assistant", "content": response},
    {"role": "user", "content": json.dumps(result)},
])
```

## Step 6 · Decide whether to continue

**Same as notebook 2.**

- Stop if:
  - the action is `terminate` → print the final answer and exit (`break`);
  - `max_iterations` is reached → safety net against endless loops.
- Otherwise → back to step 1 with the updated memory.

In the loop this is the `break` on `terminate` plus the `while iterations < max_iterations` condition.

**Expected path for the example question** (~6 iterations):

| Iteration | Action | Why |
|---|---|---|
| 1 | `small_angle_period()` | theoretical reference $T_0$ |
| 2 | `simulate_pendulum(theta0_deg=10)` | first amplitude |
| 3 | `measure_period("sim_1")` | period at 10° |
| 4 | `simulate_pendulum(theta0_deg=80)` | second amplitude |
| 5 | `measure_period("sim_2")` | period at 80° |
| 6 | `terminate(...)` | comparison + physical explanation |

The order may change: the agent decides. That is exactly the difference from the quasi-agent of notebook 1.

## Putting it all together: the agent loop

All six steps in one `while` loop. Try:
- *"How does the period of the pendulum change from 10° to 80°? Compare it with the small-angle approximation."*
- *"How does the period of a 2 m long pendulum change between 20° and 120°?"*

> **What to expect from a 2B model:** it sometimes gets the format wrong, repeats an action or forgets to measure. That's normal: the parser returns an error, the agent usually corrects itself, and `max_iterations` stops the loop from running forever.

In [ ]:
max_iterations = 12   # safety net: stop even if the agent never calls terminate
iterations = 0
simulations.clear()   # no old simulations

# The task comes from the user and is the first message in memory
user_task = input("What would you like me to do? ")
memory = [{"role": "user", "content": user_task}]

while iterations < max_iterations:
    print(f"\n--- Iteration {iterations + 1} ---")

    # 1. Construct prompt: rules + memory
    prompt = agent_rules + memory

    # 2. Generate response: the LLM chooses the next action
    print("Agent thinking...")
    response = generate_response(prompt)
    print(f"Agent response: {response}")

    # 3. Parse response -> {"tool_name": ..., "args": ...}
    action = parse_action(response)
    tool, args = action["tool_name"], action["args"]

    # 4. Execute action  (+ 6. stop on terminate)
    if tool == "terminate":
        print("\n=== Final answer ===")
        print(args.get("message", ""))
        break
    try:
        if tool == "simulate_pendulum":
            result = {"result": simulate_pendulum(**args)}
        elif tool == "measure_period":
            result = {"result": measure_period(args["sim_id"])}
        elif tool == "small_angle_period":
            result = {"result": small_angle_period(**args)}
        elif tool == "error":
            result = {"error": args["message"]}
        else:
            result = {"error": "Unknown action: " + tool}
    except Exception as e:
        result = {"error": f"Error while running {tool}: {e}"}

    print(f"Action result: {result}")

    # 5. Update memory: action + result
    memory.extend([
        {"role": "assistant", "content": response},
        {"role": "user", "content": json.dumps(result)},
    ])

    # 6. Continue? (terminate handled above; here: count iterations)
    iterations += 1
else:
    print(f"\nStopped: reached the maximum of {max_iterations} iterations.")


--- Iteration 1 ---
Agent thinking...
Agent response: <Stop and think step by step. Parameters map to args. Insert a rich description of your step by step thoughts here.>

I need to find the amplitude (in degrees) at which the period of a simple pendulum is 10% longer than the small-angle period. The small-angle period is T0 = 2π√(L/g). The observed period T for a finite amplitude is longer. I need to find the amplitude θ0 such that T = 1.10 × T0.

I'll use the simulate_pendulum tool to simulate the pendulum at various amplitudes, then measure_period to get the period, and compare with the small-angle period. But I need to be systematic: I'll first compute the small-angle period for a given length, then simulate for different amplitudes and measure their periods. However, the tools don't let me search over amplitudes directly. I'll need to simulate for a few candidate amplitudes and measure, then reason about the answer.

Actually, I can use the small_angle_period tool to get T0 for a

## What happened in this run

Task given to the agent (exercise 1): *"Find the amplitude at which the period is 10% longer than the small-angle period."*

**The good part: the reasoning**
- The model understood the task: find $\theta_0$ such that $T = 1.10\,T_0$.
- It made a sensible plan: get $T_0$ with `small_angle_period`, then simulate increasing amplitudes and measure their periods.
- It chose the right first tool with the right argument: `small_angle_period(length_m=1.0)`.

**The problem: the format**

| Expected | What the model wrote |
|---|---|
| ```` ```action ```` … ```` ``` ```` (Markdown code block) | `<action>` … `</action>` (XML-style tags) |
| its own thoughts | first copied the placeholder line `<Stop and think step by step…>` literally |

Why it failed, step by step:
1. `extract_markdown_block` looks for ```` ```action ````, then for any ```` ``` ````, and finds neither → returns the **whole text**.
2. `json.loads` on the whole text fails → `parse_action` returns the `error` action.
3. The error message (*"Invalid JSON response…"*) does **not say what was wrong** → in iteration 2 the model makes **exactly the same mistake**.
4. The run was stopped during iteration 3; without stopping it, the agent would have used all 12 iterations without calling a single tool.

**Probable cause:** the format template in `agent_rules` starts with a line in angle brackets (`<Stop and think…>`), which invites XML-style tags; the model then wrapped the action in `<action>` tags too.

**Lessons**
- Small models **drift from the required format** even when the reasoning is correct.
- **Error messages must be actionable:** tell the model what was wrong and show the correct format; otherwise it cannot correct itself.
- The **parser can be tolerant:** accept common variations (here `<action>…</action>`) instead of rejecting them.
- Always **look at the raw responses** when an agent loops: the `Agent response:` prints made this diagnosis possible.

**Possible fixes** (try them, then re-run the loop):

1. Tolerant parser: accept `<action>` tags too, at the start of `extract_markdown_block`:
   ```python
   if f"<{block_type}>" in response:
       return response.split(f"<{block_type}>", 1)[1].split(f"</{block_type}>", 1)[0].strip()
   ```
2. Actionable error message in `parse_action`:
   ```python
   "Invalid JSON response. Put the JSON inside a ```action code block, e.g.\n"
   "```action\n{\"tool_name\": \"small_angle_period\", \"args\": {}}\n```"
   ```
3. Clearer template in `agent_rules`: replace the `<Stop and think…>` line with plain text, e.g. `Thoughts: <your step-by-step reasoning>`, and add *"Do not use XML tags."*

For reference, the exact answer to this task (elliptic integral) is $\theta_0 \approx 69.3°$.

## Exercises

1. **Iterative search:** ask *"Find the amplitude at which the period is 10% longer than the small-angle period."* The agent must try several amplitudes and narrow the range: a truly agentic task.
2. **New tool: plot.** `save_plot(sim_id)` that saves $\theta(t)$ to a `.png` file and returns the file name.
3. **Limits:** set `max_iterations = 3` and see what happens. Then remove the rule *"never do calculations in your head"* and compare the answers.